# Visualize Turn Failure Cases (Understeer / Oversteer)

For each selected sample: **top row** = 6 camera images, **bottom** = BEV with GT (green) vs predicted (red) trajectory, ego vehicle, agent boxes, and drivable area.

In [ ]:
import os, sys, json, re, pickle, math
from typing import Dict, List, Optional
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle
from PIL import Image

from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

# ── paths ──
NUSCENES_ROOT = "/data/jykim/projects/OpenDriveVLA/data/nuscenes/"
ANNO_PKL      = os.path.join(NUSCENES_ROOT, "nuscenes2d_ego_temporal_infos_val_with_command_desc.pkl")
PRED_PATH     = "/data/jykim/projects/SpaceDrive/workspace/spacedrive_plus_qwen/_results_planning_only/"
SAVE_DIR      = "/data/jykim/projects/SpaceDrive/workspace/understeer_vis/"
os.makedirs(SAVE_DIR, exist_ok=True)

# map API lives next to the evaluation scripts
sys.path.insert(0, "/data/jykim/projects/SpaceDrive/scripts/evaluation")
from map_api import NuScenesMap

CAMERA_ORDER = [
    "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
    "CAM_BACK_LEFT",  "CAM_BACK",  "CAM_BACK_RIGHT",
]

# ── tunables ──
UNDERSTEER_MARGIN_DEG = 3.0
TURN_ANGLE_THRESH_DEG = 10.0
TOP_K = 5

print("Setup OK")

## 1. Load Data & Classify Samples

In [ ]:
# ── helpers (reused from analyze_turn_understeer.py) ──

def extract_coords_from_text(traj_text: str) -> np.ndarray:
    full_match = re.search(
        r"\[<POS_INDICATOR>\(([\d\.\-]+, [\d\.\-]+)\)\s*"
        r"([\s\S]*<POS_INDICATOR>\([\d\.\-]+, [\d\.\-]+\)\s*)*",
        traj_text,
    )
    if full_match is None:
        coords = re.findall(r"\(\s*[-+]?\d*\.?\d+\s*,\s*[-+]?\d*\.?\d+\s*\)", traj_text)
        if not coords:
            return np.empty((0, 2), dtype=np.float32)
    else:
        coords = re.findall(r"\(\+?[\d\.\-]+, \+?[\d\.\-]+\)", full_match.group(0))
    parsed = [tuple(map(float, re.findall(r"-?\d+\.\d+|-?\d+", c))) for c in coords]
    if not parsed:
        return np.empty((0, 2), dtype=np.float32)
    return np.array(parsed, dtype=np.float32)


def traj_turn_angle_deg(traj_xy: np.ndarray) -> float:
    if traj_xy is None or len(traj_xy) < 3:
        return 0.0
    diffs = np.diff(traj_xy[:, :2], axis=0)
    valid = np.linalg.norm(diffs, axis=1) > 1e-6
    diffs = diffs[valid]
    if len(diffs) < 2:
        return 0.0
    headings = np.arctan2(diffs[:, 1], diffs[:, 0])
    headings = np.unwrap(headings)
    return float(np.degrees(headings[-1] - headings[0]))


def is_turn_by_goal(command_desc, command_id=None, mode="strict"):
    if isinstance(command_desc, str):
        low = command_desc.lower()
        for kw in ("turning left", "turning right", "turning sharp left", "turning sharp right"):
            if kw in low:
                return True
    return False


def compute_l2_3s(pred_xy, gt_xy):
    n = min(len(pred_xy), len(gt_xy), 6)
    if n == 0:
        return float("inf")
    return float(np.mean(np.linalg.norm(pred_xy[:n] - gt_xy[:n], axis=1)))


# ── load ──
print("Loading annotation PKL …")
with open(ANNO_PKL, "rb") as f:
    anno_data = pickle.load(f)
infos = anno_data["infos"]
info_by_token = {d["token"]: d for d in infos}

print("Loading predictions …")
preds = {}
for d in infos:
    token = d["token"]
    pf = os.path.join(PRED_PATH, token)
    if not os.path.exists(pf):
        continue
    try:
        with open(pf) as f:
            pred_data = json.load(f)
        coords = extract_coords_from_text(pred_data[0]["A"])
        if coords.shape[0] >= 6:
            preds[token] = coords[:6, :2]
    except Exception:
        continue

print(f"Loaded {len(preds)} predictions out of {len(infos)} annotations")

# ── classify ──
records = []
for token, pred_xy in preds.items():
    d = info_by_token[token]
    gt_xy = d["gt_planning"][0, :6, :2]
    mask = d["gt_planning_mask"][0]
    if not mask.all():
        continue

    cmd_turn = is_turn_by_goal(d.get("gt_planning_command_desc"), d.get("gt_planning_command"))
    gt_angle = traj_turn_angle_deg(gt_xy)
    pred_angle = traj_turn_angle_deg(pred_xy)
    is_turn = cmd_turn or abs(gt_angle) >= TURN_ANGLE_THRESH_DEG

    if not is_turn:
        continue

    l2_3s = compute_l2_3s(pred_xy, gt_xy)
    abs_pred, abs_gt = abs(pred_angle), abs(gt_angle)

    if (abs_pred + UNDERSTEER_MARGIN_DEG) < abs_gt:
        steer_cat = "under_steer"
    elif (abs_pred - UNDERSTEER_MARGIN_DEG) > abs_gt:
        steer_cat = "over_steer"
    else:
        steer_cat = "good"

    records.append({
        "token": token,
        "steer_cat": steer_cat,
        "l2_3s": l2_3s,
        "gt_angle": gt_angle,
        "pred_angle": pred_angle,
        "command_desc": d.get("gt_planning_command_desc", ""),
    })

print(f"\nTurn samples with predictions: {len(records)}")
for cat in ("under_steer", "over_steer", "good"):
    n = sum(1 for r in records if r["steer_cat"] == cat)
    print(f"  {cat}: {n}")

## 2. Select Samples

In [ ]:
under_steer = sorted([r for r in records if r["steer_cat"] == "under_steer"],
                      key=lambda r: r["l2_3s"], reverse=True)[:TOP_K]
over_steer  = sorted([r for r in records if r["steer_cat"] == "over_steer"],
                      key=lambda r: r["l2_3s"], reverse=True)[:TOP_K]
good        = sorted([r for r in records if r["steer_cat"] == "good"],
                      key=lambda r: r["l2_3s"])[:TOP_K]

selected = under_steer + over_steer + good

print(f"Selected {len(selected)} samples:")
for r in selected:
    print(f"  [{r['steer_cat']:>12s}]  L2_3s={r['l2_3s']:.3f}  "
          f"gt_angle={r['gt_angle']:+.1f}°  pred_angle={r['pred_angle']:+.1f}°  "
          f"cmd={r['command_desc'][:40]}")

## 3. Load nuScenes & Map

In [ ]:
nusc = NuScenes(version="v1.0-trainval", dataroot=NUSCENES_ROOT, verbose=True)

nusc_maps = {
    loc: NuScenesMap(dataroot=NUSCENES_ROOT, map_name=loc)
    for loc in ["boston-seaport", "singapore-hollandvillage",
                "singapore-onenorth", "singapore-queenstown"]
}
print("Maps loaded:", list(nusc_maps.keys()))

## 4. Visualization Functions

In [ ]:
# ─── BEV helpers ───

EGO_LENGTH, EGO_WIDTH = 4.084, 1.85
BEV_FRONT, BEV_REAR = 30.0, 10.0   # meters
BEV_SIDE = 15.0                      # meters each side
BEV_RES  = 0.1                       # m/pixel
BEV_W = int((2 * BEV_SIDE) / BEV_RES)    # 300 px
BEV_H = int((BEV_FRONT + BEV_REAR) / BEV_RES)  # 400 px


def ego_to_bev_px(x, y):
    """Convert ego-frame (x=forward, y=left) → BEV pixel (col, row).
    BEV image: top=front, bottom=rear, left→left, right→right.
    """
    col = (BEV_SIDE + y) / BEV_RES       # y=0 → centre column
    row = (BEV_FRONT - x) / BEV_RES      # x=0 → BEV_FRONT from top
    return col, row


def rotated_box_corners(cx, cy, yaw, length, width):
    """Return 4 corners of a rotated box in ego frame."""
    cos_a, sin_a = np.cos(yaw), np.sin(yaw)
    dx = np.array([length / 2, length / 2, -length / 2, -length / 2])
    dy = np.array([width / 2, -width / 2, -width / 2, width / 2])
    xs = cx + cos_a * dx - sin_a * dy
    ys = cy + sin_a * dx + cos_a * dy
    return xs, ys


def get_drivable_mask(info, nusc_maps_dict):
    """Rasterize drivable area into the BEV crop.

    get_map_mask returns a *symmetric* mask centred on the ego.  Internally
    (traced through _get_layer_polygon → _polygon_geom_to_mask →
    mask_for_polygons / cv2.fillPoly) the rasterisation maps:
        local_y  (ego-forward, +y = ahead)  →  mask row   (row 0 = rear)
        local_x  (ego-right,   +x = right)  →  mask col   (col 0 = left)

    Our BEV viewport is asymmetric (front 30 m, rear 10 m, side ±15 m),
    so we generate a symmetric mask large enough to cover it, then crop
    the sub-region that corresponds to our viewport.

    Returns (BEV_H, BEV_W) bool mask aligned with the bev_img array
    used by draw_bev (row 0 = rear, row BEV_H-1 = front, matching
    matplotlib origin='lower').
    """
    location = info["location"]
    nusc_map = nusc_maps_dict[location]
    e2g_r = Quaternion(info["ego2global_rotation"]).rotation_matrix
    e2g_t = np.array(info["ego2global_translation"])
    yaw = math.degrees(Quaternion(matrix=e2g_r).yaw_pitch_roll[0])

    # Symmetric patch with 1 m margin on each side so the crop stays
    # within bounds.  Half-extent >= max(BEV_FRONT, BEV_REAR) + margin.
    margin = 1.0
    half_long = max(BEV_FRONT, BEV_REAR) + margin   # 31 m
    half_lat  = BEV_SIDE + margin                    # 16 m
    sym_long  = half_long * 2                        # 62 m total
    sym_lat   = half_lat  * 2                        # 32 m total
    sym_h_px  = int(sym_long / BEV_RES)              # 620 rows
    sym_w_px  = int(sym_lat  / BEV_RES)              # 320 cols

    patch_box   = (e2g_t[0], e2g_t[1], sym_long, sym_lat)
    canvas_size = (sym_h_px, sym_w_px)

    full_mask = nusc_map.get_map_mask(
        patch_box, yaw, ["drivable_area"], canvas_size=canvas_size
    ).squeeze(0)  # (sym_h_px, sym_w_px)

    # Mask row i represents forward position:
    #   fwd_m = -half_long + i * BEV_RES
    # Row for a given fwd_m:
    #   row = (fwd_m + half_long) / BEV_RES
    row_start = int((-BEV_REAR  + half_long) / BEV_RES)  # fwd=-10 → row 210
    row_end   = int(( BEV_FRONT + half_long) / BEV_RES)  # fwd=+30 → row 610
    col_start = int((-BEV_SIDE  + half_lat)  / BEV_RES)  # lat=-15 → col 10
    col_end   = int(( BEV_SIDE  + half_lat)  / BEV_RES)  # lat=+15 → col 310

    cropped = full_mask[row_start:row_end, col_start:col_end]
    return cropped.astype(bool)


def draw_bev(ax, info, pred_xy, nusc_maps_dict):
    """Draw BEV on the given matplotlib Axes."""
    gt_xy = info["gt_planning"][0, :6, :2]

    # ── drivable area ──
    try:
        drivable = get_drivable_mask(info, nusc_maps_dict)
        bev_img = np.ones((BEV_H, BEV_W, 3), dtype=np.float32) * 0.92
        bev_img[drivable]  = [0.82, 0.87, 0.82]
        bev_img[~drivable] = [0.95, 0.93, 0.90]
    except Exception:
        bev_img = np.ones((BEV_H, BEV_W, 3), dtype=np.float32) * 0.92

    ax.imshow(bev_img, extent=[-BEV_SIDE, BEV_SIDE, -BEV_REAR, BEV_FRONT], origin="lower")

    # ── surrounding agent boxes (t=0) ──
    gt_boxes = info.get("gt_boxes")
    if gt_boxes is not None:
        for bi in range(gt_boxes.shape[0]):
            bx, by, bz, bw, bl, bh, byaw = gt_boxes[bi, :7]
            # filter to BEV range
            if abs(by) > BEV_SIDE + 5 or bx < -(BEV_REAR + 5) or bx > BEV_FRONT + 5:
                continue
            xs, ys = rotated_box_corners(bx, by, byaw, bl, bw)
            poly = plt.Polygon(np.column_stack([ys, xs]), closed=True,
                               fc=(0.55, 0.70, 0.90, 0.5), ec=(0.2, 0.3, 0.6), lw=0.8)
            ax.add_patch(poly)

    # ── ego vehicle ──
    ego_xs, ego_ys = rotated_box_corners(0, 0, 0, EGO_LENGTH, EGO_WIDTH)
    ego_poly = plt.Polygon(np.column_stack([ego_ys, ego_xs]), closed=True,
                            fc=(0.3, 0.3, 0.3, 0.7), ec="black", lw=1.5)
    ax.add_patch(ego_poly)

    # ── trajectories ──
    # prepend origin (ego position at t=0)
    gt_full  = np.vstack([[0, 0], gt_xy])
    pred_full = np.vstack([[0, 0], pred_xy])

    ax.plot(gt_full[:, 1],  gt_full[:, 0],  "o-", color="#2e7d32", ms=5, lw=2.2,
            label="GT", zorder=5)
    ax.plot(pred_full[:, 1], pred_full[:, 0], "s-", color="#c62828", ms=5, lw=2.2,
            label="Pred", zorder=5)

    # waypoint indices
    for i in range(1, gt_full.shape[0]):
        ax.annotate(str(i), (gt_full[i, 1], gt_full[i, 0]),
                    fontsize=6, color="#2e7d32", fontweight="bold",
                    ha="left", va="bottom")
    for i in range(1, pred_full.shape[0]):
        ax.annotate(str(i), (pred_full[i, 1], pred_full[i, 0]),
                    fontsize=6, color="#c62828", fontweight="bold",
                    ha="right", va="top")

    # ── formatting ──
    ax.set_xlim(-BEV_SIDE, BEV_SIDE)
    ax.set_ylim(-BEV_REAR, BEV_FRONT)
    ax.set_aspect("equal")
    ax.set_xlabel("Lateral (m)", fontsize=9)
    ax.set_ylabel("Longitudinal (m)", fontsize=9)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

    # direction arrow
    ax.annotate("", xy=(0, BEV_FRONT - 1), xytext=(0, BEV_FRONT - 4),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    ax.text(0.5, BEV_FRONT - 2, "front", fontsize=7, color="gray", ha="left")


def load_camera_images(info, nusc):
    """Load the 6 camera images for a sample (using nusc to resolve paths)."""
    imgs = {}
    for cam_name in CAMERA_ORDER:
        cam_info = info["cams"][cam_name]
        data_path = cam_info["data_path"]
        # resolve path
        for prefix in ("./data/nuscenes/", "data/nuscenes/"):
            if data_path.startswith(prefix):
                data_path = os.path.join(NUSCENES_ROOT, data_path[len(prefix):])
                break
        if os.path.exists(data_path):
            imgs[cam_name] = Image.open(data_path)
        else:
            imgs[cam_name] = Image.new("RGB", (1600, 900), (128, 128, 128))
    return imgs


print("BEV helpers defined ✓")

## 5. Generate & Save Figures

In [ ]:
from matplotlib.gridspec import GridSpec

CAM_LABELS = {
    "CAM_FRONT_LEFT":  "Front-Left",
    "CAM_FRONT":       "Front",
    "CAM_FRONT_RIGHT": "Front-Right",
    "CAM_BACK_LEFT":   "Back-Left",
    "CAM_BACK":        "Back",
    "CAM_BACK_RIGHT":  "Back-Right",
}

saved_files = []

for rec in selected:
    token = rec["token"]
    info  = info_by_token[token]
    pred_xy = preds[token]

    imgs = load_camera_images(info, nusc)

    # ── figure layout ──
    fig = plt.figure(figsize=(24, 14))
    gs = GridSpec(2, 6, figure=fig, height_ratios=[1, 1.4],
                  hspace=0.15, wspace=0.04)

    # top row: 6 cameras
    for ci, cam_name in enumerate(CAMERA_ORDER):
        ax = fig.add_subplot(gs[0, ci])
        ax.imshow(imgs[cam_name])
        ax.set_title(CAM_LABELS[cam_name], fontsize=10, fontweight="bold")
        ax.axis("off")

    # bottom row: BEV spanning columns 1-4 (centred)
    ax_bev = fig.add_subplot(gs[1, 1:5])
    draw_bev(ax_bev, info, pred_xy, nusc_maps)

    # suptitle
    cat = rec["steer_cat"]
    l2 = rec["l2_3s"]
    cmd = rec["command_desc"].strip()[:60] if rec["command_desc"] else ""
    fig.suptitle(
        f"[{cat.upper().replace('_',' ')}]  L2_3s={l2:.3f}m   "
        f"GT angle={rec['gt_angle']:+.1f}°  Pred angle={rec['pred_angle']:+.1f}°\n"
        f"Token: {token}   Cmd: {cmd}",
        fontsize=13, fontweight="bold", y=0.98,
    )

    # save
    fname = f"{token}_{cat}_{l2:.3f}.png"
    fpath = os.path.join(SAVE_DIR, fname)
    fig.savefig(fpath, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    saved_files.append(fpath)
    print(f"  Saved: {fname}")

print(f"\nDone — {len(saved_files)} figures saved to {SAVE_DIR}")

## 6. Preview a Sample

In [ ]:
if saved_files:
    from IPython.display import display, Image as IPImage
    display(IPImage(filename=saved_files[0], width=1200))